# Protobuf Basics — 08: Nested Protobuf Messages

Protobuf's repeated and nested message fields map to Spark's
ArrayType and StructType, enabling rich hierarchical data.

Topics: nested messages in Spark, repeated fields as arrays,
map fields, oneof variants, flattening for analytics.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/protobuf_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('protobuf-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Nested Message → StructType



In [ ]:

print("""
Protobuf nested message maps to Spark StructType:

  message Address {          →  StructType([
    string city    = 1;            StructField("city",    StringType()),
    string country = 2;            StructField("country", StringType()),
    string zip     = 3;            StructField("zip",     StringType()),
  }                          ])

  message Order {            →  StructType([
    int64   order_id = 1;          StructField("order_id",  LongType()),
    Address address  = 2;          StructField("address", StructType([...])),
    ...                            ...
  }                          ])

Accessing in Spark:
  df.select("order_id", "address.city", "address.country")
""")


## Step 2 — Repeated Field → ArrayType



In [ ]:

print("""
Protobuf repeated field maps to Spark ArrayType:

  message Order {
    repeated LineItem items = 6;   →  ArrayType(StructType([...]))
    repeated string   tags  = 7;   →  ArrayType(StringType())
  }

  message LineItem {
    string sku      = 1;
    double price    = 2;
    int32  quantity = 3;
  }

Accessing in Spark:
  # Explode to one row per item
  df.select("order_id", F.explode("items").alias("item")) \
    .select("order_id", "item.sku", "item.price", "item.quantity")

  # Aggregate array
  df.withColumn("item_count", F.size("items")) \
    .withColumn("total", F.aggregate("items", F.lit(0.0),
                                     lambda acc,x: acc + x.price * x.quantity))
""")


## Step 3 — Map Fields → MapType



In [ ]:

print("""
Protobuf map field maps to Spark MapType:

  message Order {
    map<string, string> metadata = 10;   →  MapType(StringType(), StringType())
    map<string, double> metrics  = 11;   →  MapType(StringType(), DoubleType())
  }

Note: Protobuf maps are unordered key-value pairs.
Keys must be scalar types (string/int/etc.) — NOT float or bytes.

Accessing in Spark:
  df.select(
      F.col("metadata").getItem("session").alias("session"),
      F.col("metrics").getItem("score").alias("score"),
  )

  # Explode map to key-value rows
  df.select("order_id", F.explode("metadata").alias("key","value"))
""")


## Step 4 — Simulating Nested Protobuf in Spark



In [ ]:

from pyspark.sql.types import *

# Simulate what from_protobuf produces for a nested Order message
nested_schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("address", StructType([
        StructField("city",    StringType()),
        StructField("country", StringType()),
        StructField("zip",     StringType()),
    ])),
    StructField("items", ArrayType(StructType([
        StructField("sku",      StringType()),
        StructField("price",    DoubleType()),
        StructField("quantity", IntegerType()),
    ]))),
    StructField("tags",     ArrayType(StringType())),
    StructField("metadata", MapType(StringType(), StringType())),
    StructField("status",   StringType()),
])

import random; random.seed(42)
rows = []
CTRS=["US","UK","DE","FR","JP"]
for i in range(5000):
    rows.append((
        i+1, random.randint(1,500),
        (random.choice(["NYC","LA","Berlin","London","Paris"]),
         random.choice(CTRS), f"{10000+i}"),
        [(f"SKU{j}", round(random.uniform(5,100),2), random.randint(1,5))
         for j in range(random.randint(1,4))],
        random.sample(["vip","mobile","promo","returning"],random.randint(0,3)),
        {"session":f"s_{i}","ab":f"v{i%3+1}"},
        random.choice(["confirmed","shipped","delivered"])
    ))

df = spark.createDataFrame(rows, nested_schema)
print(f"Nested Protobuf-like DataFrame: {df.count():,} rows")
df.printSchema()
df.show(3, truncate=False)


## Step 5 — Flattening Nested Protobuf



In [ ]:

# Flatten for analytics storage
df_flat = df.select(
    "order_id","customer_id",
    F.col("address.city").alias("city"),
    F.col("address.country").alias("country"),
    F.size("items").alias("item_count"),
    F.aggregate("items", F.lit(0.0),
                lambda acc,x: acc + x["price"]*x["quantity"]).alias("total_value"),
    F.size("tags").alias("tag_count"),
    F.col("metadata").getItem("ab").alias("ab_test"),
    "status"
)
print("Flattened for analytics:")
df_flat.show(5)
df_flat.write.mode("overwrite").option("compression","zstd").parquet(f"{DATA_DIR}/nested_flat")
print(f"Saved as Parquet: {spark.read.parquet(f'{DATA_DIR}/nested_flat').count():,} rows")


## Summary



In [ ]:

print("""
Protobuf → Spark type mapping:
  message                    → StructType
  repeated scalar            → ArrayType(ScalarType)
  repeated message           → ArrayType(StructType)
  map<string, scalar>        → MapType(StringType, ScalarType)
  oneof                      → StructType (all fields nullable, at most one non-null)
  enum                       → IntegerType (enum ordinal value)
  google.protobuf.Timestamp  → StructType(seconds:long, nanos:int)
  google.protobuf.StringValue→ StringType (nullable wrapper type)

Accessing:
  nested: df.select("parent.child.grandchild")
  array:  F.explode("items"), F.size("items"), F.aggregate(...)
  map:    F.col("map").getItem("key"), F.explode("map")
  oneof:  filter on non-null variant field
""")
